# System 2107: Aligned 5-minute Dataset
Implement the approved alignment using electrical, POA, and environment inputs only. Meter stays separate for later validation.

- Preserve electrical timestamps and all inverter columns within the three-input overlap.
- POA joins exactly. Environment maps only from t to t/t+5/t+10; absent source intervals and missing cells stay missing.
- Plant AC power requires all 24 channels. Availability/QC flags do not remove rows or alter measurements.
- Keep local wall-clock timestamps and mark ambiguous fall 01:xx. No UTC conversion, interpolation, or additional cleaning.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data/2107/processed').is_dir():
    ROOT = ROOT.parent
DATA = ROOT / 'data/2107/processed'
paths = {name: DATA / f'2107_{name}_clean.parquet'
         for name in ['electrical', 'irradiance', 'environment']}
qc_paths = {name: DATA / f'2107_{name}_qc_events.parquet'
            for name in ['electrical', 'irradiance']}
OUTPUT = DATA / '2107_model_5min.parquet'
BATCH_SIZE = 25_000

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

inputs = list(paths.values()) + list(qc_paths.values())
assert all(path.is_file() for path in inputs)
input_hashes = {str(path.relative_to(ROOT)): sha256(path) for path in inputs}
electrical_file = pq.ParquetFile(paths['electrical'])
electrical_schema = electrical_file.schema_arrow
inverter_columns = [c for c in electrical_schema.names if c != 'measured_on']
ac_columns = [c for c in inverter_columns if '_ac_power_' in c]
assert len(ac_columns) == 24
assert {int(c.split('_')[1]) for c in ac_columns} == set(range(1, 25))

source_times = {}
for name, path in paths.items():
    stamps = pq.read_table(path, columns=['measured_on']).column(0).to_pandas()
    index = pd.DatetimeIndex(pd.to_datetime(stamps, format='%Y-%m-%d %H:%M:%S', errors='raise'))
    assert index.is_unique and index.is_monotonic_increasing
    minutes = 15 if name == 'environment' else 5
    assert ((index.minute % minutes == 0) & (index.second == 0)).all()
    source_times[name] = index

bounds = pd.DataFrame([dict(dataset=name, rows=len(index), first=index[0], last=index[-1])
                       for name, index in source_times.items()])
overlap_start = max(index[0] for index in source_times.values())
overlap_end = min(index[-1] for index in source_times.values())
assert overlap_start <= overlap_end
base_index = source_times['electrical']
expected_index = base_index[(base_index >= overlap_start) & (base_index <= overlap_end)]
display(bounds)
print('Three-input overlap:', overlap_start, 'to', overlap_end)
print('Expected electrical-base rows:', len(expected_index))

Three-input overlap: 2017-12-01 00:00:00 to 2023-10-31 23:45:00
Expected electrical-base rows: 622294


,dataset,rows,first,last
0,electrical,632952,2017-11-01 00:00:00,2023-11-07 23:55:00
1,irradiance,531019,2017-11-01 07:10:00,2023-11-01 23:55:00
2,environment,206008,2017-12-01 00:00:00,2023-10-31 23:45:00


## Input mappings and schema
Overlap uses the first/last actual source labels, including the last environment label at 23:45; it is not extended to 23:55. Missing rows inside the overlap are retained.

`model_inputs_available` requires POA, temperature, and wind speed, independently of electrical telemetry and wind direction. QC flags mean membership in a source QC ledger, regardless of flag class.

In [2]:
irradiance = pq.read_table(paths['irradiance']).to_pandas()
assert len(irradiance.columns) == 2
poa_column = next(c for c in irradiance.columns if c != 'measured_on')
irradiance.index = source_times['irradiance']
poa = irradiance[poa_column]
environment = pq.read_table(paths['environment']).to_pandas()
environment.index = source_times['environment']
env_columns = ['ambient_temperature_c', 'wind_speed_m_s', 'wind_direction_deg']
assert list(environment.columns) == ['measured_on'] + env_columns
environment = environment[env_columns]
qc_times = {name: set(pq.read_table(path, columns=['measured_on']).column(0).to_pylist())
            for name, path in qc_paths.items()}

availability_columns = ['poa_available', 'environment_available', 'all_inverter_ac_available',
                        'model_inputs_available', 'electrical_qc_flag', 'irradiance_qc_flag',
                        'dst_fall_ambiguous']
model_columns = ['poa_w_m2'] + env_columns
output_columns = (['measured_on'] + model_columns + availability_columns +
                  ['plant_ac_power_kw', 'inverter_ac_available_count'] + inverter_columns)
assert len(output_columns) == len(set(output_columns))
output_schema = pa.schema(
    [electrical_schema.field('measured_on')] +
    [pa.field(c, pa.float64()) for c in model_columns] +
    [pa.field(c, pa.bool_()) for c in availability_columns] +
    [pa.field('plant_ac_power_kw', pa.float64()), pa.field('inverter_ac_available_count', pa.int8())] +
    [electrical_schema.field(c) for c in inverter_columns])

# Pacific fall transitions follow the first-November-Sunday rule in these years.
assert expected_index[0].year >= 2007
fall_dates = []
for year in range(expected_index[0].year, expected_index[-1].year + 1):
    first = pd.Timestamp(year=year, month=11, day=1)
    fall_dates.append(first + pd.Timedelta(days=(6 - first.weekday()) % 7))

def align_batch(source):
    source = source.reset_index(drop=True)
    index = pd.DatetimeIndex(pd.to_datetime(source['measured_on'], format='%Y-%m-%d %H:%M:%S'))
    result = source.copy()
    result['poa_w_m2'] = poa.reindex(index).to_numpy()
    # Exact interval anchors prevent carrying values across absent environment rows.
    anchors = index.floor('15min')
    mapped = environment.reindex(anchors)
    result[env_columns] = mapped.to_numpy()
    count = source[ac_columns].notna().sum(axis=1)
    result['inverter_ac_available_count'] = count.astype('int8')
    result['plant_ac_power_kw'] = source[ac_columns].sum(axis=1, min_count=24)
    result['poa_available'] = result['poa_w_m2'].notna()
    result['environment_available'] = result[['ambient_temperature_c', 'wind_speed_m_s']].notna().all(axis=1)
    result['all_inverter_ac_available'] = count.eq(24)
    result['model_inputs_available'] = result['poa_available'] & result['environment_available']
    for name in ['electrical', 'irradiance']:
        result[f'{name}_qc_flag'] = source['measured_on'].isin(qc_times[name])
    ambiguous = np.zeros(len(index), dtype=bool)
    for day in fall_dates:
        ambiguous |= (index >= day + pd.Timedelta(hours=1)) & (index < day + pd.Timedelta(hours=2))
    result['dst_fall_ambiguous'] = ambiguous
    return result[output_columns]

print('Output columns:', output_columns)

Output columns: ['measured_on', 'poa_w_m2', 'ambient_temperature_c', 'wind_speed_m_s', 'wind_direction_deg', 'poa_available', 'environment_available', 'all_inverter_ac_available', 'model_inputs_available', 'electrical_qc_flag', 'irradiance_qc_flag', 'dst_fall_ambiguous', 'plant_ac_power_kw', 'inverter_ac_available_count', 'inv_01_dc_current_inv_149579', 'inv_01_dc_voltage_inv_149580', 'inv_01_ac_current_inv_149581', 'inv_01_ac_voltage_inv_149582', 'inv_01_ac_power_inv_149583', 'inv_02_dc_current_inv_149584', 'inv_02_dc_voltage_inv_149585', 'inv_02_ac_current_inv_149586', 'inv_02_ac_voltage_inv_149587', 'inv_02_ac_power_inv_149588', 'inv_03_dc_current_inv_149589', 'inv_03_dc_voltage_inv_149590', 'inv_03_ac_current_inv_149591', 'inv_03_ac_voltage_inv_149592', 'inv_03_ac_power_inv_149593', 'inv_04_dc_current_inv_149594', 'inv_04_dc_voltage_inv_149595', 'inv_04_ac_current_inv_149596', 'inv_04_ac_voltage_inv_149597', 'inv_04_ac_power_inv_149598', 'inv_05_dc_current_inv_149599', 'inv_05_dc_v

## Generate the single modelling dataset
Write bounded batches and preserve source order. No rows are filtered by availability or QC. Rerunning replaces only this derived output.

In [3]:
assert {str(path.relative_to(ROOT)): sha256(path) for path in inputs} == input_hashes
written_rows = 0
written_groups = 0
with pq.ParquetWriter(OUTPUT, output_schema, compression='snappy') as writer:
    for batch in electrical_file.iter_batches(batch_size=BATCH_SIZE):
        source = batch.to_pandas()
        index = pd.DatetimeIndex(pd.to_datetime(source['measured_on'], format='%Y-%m-%d %H:%M:%S'))
        source = source.loc[(index >= overlap_start) & (index <= overlap_end)]
        if source.empty:
            continue
        aligned = align_batch(source)
        table = pa.Table.from_pandas(aligned, schema=output_schema, preserve_index=False)
        writer.write_table(table, row_group_size=len(aligned))
        written_rows += len(aligned)
        written_groups += 1
assert written_rows == len(expected_index)
print('Created:', OUTPUT.relative_to(ROOT), 'rows:', written_rows)

Created: data\2107\processed\2107_model_5min.parquet rows: 622294


## Validate saved output
Re-read electrical batches and each output row group. Independently expand environment observations to exactly t/t+5/t+10 for validation. Compare every measurement, flag, timestamp, and column type; verify unchanged input hashes.

In [4]:
expanded_environment = pd.concat([
    environment.set_axis(environment.index + pd.Timedelta(minutes=offset))
    for offset in [0, 5, 10]
]).sort_index()
assert expanded_environment.index.is_unique
saved = pq.ParquetFile(OUTPUT)
assert saved.schema_arrow.remove_metadata() == output_schema.remove_metadata()
assert saved.metadata.num_rows == len(expected_index)
assert saved.num_row_groups == written_groups
verified_rows = 0
group = 0
available_counts = {'model_inputs_available': 0, 'all_inverter_ac_available': 0, 'both': 0}
indicator_counts = {c: 0 for c in availability_columns}
first_timestamp = last_timestamp = None
for batch in pq.ParquetFile(paths['electrical']).iter_batches(batch_size=BATCH_SIZE):
    original = batch.to_pandas()
    index = pd.DatetimeIndex(pd.to_datetime(original['measured_on'], format='%Y-%m-%d %H:%M:%S'))
    original = original.loc[(index >= overlap_start) & (index <= overlap_end)].reset_index(drop=True)
    if original.empty:
        continue
    result = saved.read_row_group(group).to_pandas()
    group += 1
    assert list(result.columns) == output_columns
    assert len(result) == len(original)
    pd.testing.assert_frame_equal(result[['measured_on'] + inverter_columns], original,
                                  check_exact=True, check_dtype=True)
    index = pd.DatetimeIndex(pd.to_datetime(result['measured_on'], format='%Y-%m-%d %H:%M:%S'))
    assert index.equals(expected_index[verified_rows:verified_rows+len(result)])
    verified_rows += len(result)
    assert ((index.minute % 5 == 0) & (index.second == 0)).all()
    expected_poa = pd.Series(poa.reindex(index).to_numpy(), name='poa_w_m2')
    pd.testing.assert_series_equal(result['poa_w_m2'], expected_poa, check_exact=True)
    expected_environment = expanded_environment.reindex(index).reset_index(drop=True)
    pd.testing.assert_frame_equal(result[env_columns], expected_environment, check_exact=True)
    absent_poa = ~index.isin(poa.index)
    assert result.loc[absent_poa, 'poa_w_m2'].isna().all()
    absent_environment = ~index.floor('15min').isin(environment.index)
    assert result.loc[absent_environment, env_columns].isna().all().all()
    # Expansion uses exact offsets, respecting each original missing cell independently.
    offsets = (index - index.floor('15min')).total_seconds() / 60
    assert np.isin(offsets, [0, 5, 10]).all()
    ac_count = original[ac_columns].notna().sum(axis=1).astype('int8')
    pd.testing.assert_series_equal(result['inverter_ac_available_count'], ac_count.rename('inverter_ac_available_count'))
    expected_plant = original[ac_columns].sum(axis=1, min_count=24).rename('plant_ac_power_kw')
    pd.testing.assert_series_equal(result['plant_ac_power_kw'], expected_plant, check_exact=True)
    assert result.loc[ac_count.lt(24), 'plant_ac_power_kw'].isna().all()
    expected_flags = {
        'poa_available': expected_poa.notna(),
        'environment_available': expected_environment[['ambient_temperature_c', 'wind_speed_m_s']].notna().all(axis=1),
        'all_inverter_ac_available': ac_count.eq(24),
        'electrical_qc_flag': original['measured_on'].isin(qc_times['electrical']),
        'irradiance_qc_flag': original['measured_on'].isin(qc_times['irradiance']),
    }
    expected_flags['model_inputs_available'] = expected_flags['poa_available'] & expected_flags['environment_available']
    # Independent calendar check of the ambiguous first-November-Sunday hour.
    expected_flags['dst_fall_ambiguous'] = pd.Series(
        (index.month == 11) & (index.day <= 7) & (index.dayofweek == 6) & (index.hour == 1))
    for column, expected in expected_flags.items():
        pd.testing.assert_series_equal(result[column], expected.rename(column), check_exact=True)
        assert result[column].dtype == np.dtype('bool')
        indicator_counts[column] += int(result[column].sum())
    assert result['inv_05_dc_voltage_inv_149600'].isna().all()
    for column in ['model_inputs_available', 'all_inverter_ac_available']:
        available_counts[column] += int(result[column].sum())
    available_counts['both'] += int((result['model_inputs_available'] & result['all_inverter_ac_available']).sum())
    if first_timestamp is None:
        first_timestamp = result['measured_on'].iloc[0]
    last_timestamp = result['measured_on'].iloc[-1]
assert verified_rows == len(expected_index) and group == saved.num_row_groups
assert {str(path.relative_to(ROOT)): sha256(path) for path in inputs} == input_hashes
print('PASS: exact base timestamps/order, inverter measurements, POA and bounded environment mapping.')
print('PASS: source missingness, strict plant sum/count, availability/QC/DST flags, Parquet schemas/readback.')
print('PASS: all five existing input hashes unchanged; no meter data used.')

PASS: exact base timestamps/order, inverter measurements, POA and bounded environment mapping.
PASS: source missingness, strict plant sum/count, availability/QC/DST flags, Parquet schemas/readback.
PASS: all five existing input hashes unchanged; no meter data used.


## Dataset summary
Counts retain every electrical-base row in the overlap. `model_inputs_available` does not require inverter telemetry; `both` reports joint model-input and full AC-power availability. Ambiguous fall rows are retained for later timezone decisions.

In [5]:
availability_summary = pd.DataFrame([
    {'requirement': name, 'rows': count, 'percent': 100 * count / verified_rows}
    for name, count in available_counts.items()
])
print('Rows:', verified_rows, 'start:', first_timestamp, 'end:', last_timestamp)
display(availability_summary)
display(pd.Series(indicator_counts, name='true_rows').to_frame())
print('Columns:', len(output_columns), '| inverter measurement columns:', len(inverter_columns))
print('Output:', OUTPUT.relative_to(ROOT))

Rows: 622294 start: 2017-12-01 00:00:00 end: 2023-10-31 23:45:00
Columns: 134 | inverter measurement columns: 120
Output: data\2107\processed\2107_model_5min.parquet


,requirement,rows,percent
0,model_inputs_available,526068,84.536891
1,all_inverter_ac_available,622294,100.000000
2,both,526068,84.536891


,true_rows
poa_available,529952
environment_available,617578
all_inverter_ac_available,622294
model_inputs_available,526068
electrical_qc_flag,6
irradiance_qc_flag,91
dst_fall_ambiguous,60


## Remaining limitations
- Timestamps remain Pacific local wall-clock labels; fall fold identity is unresolved.
- Environment mapping implements the approved same-interval convention, without claiming its original physical hourly support is resolved.
- Suspect values and sustained zeros retain their existing values and detailed QC ledgers. No additional cleaning, modelling, or meter alignment is performed.